## Clean Transform using PySpark

####Step 1: Read the datasets

In [0]:
customers = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/workspace/default/first_volume/customers.csv")

sales = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/workspace/default/first_volume/sales.csv")

products = spark.read.option("header", "true").option("inferSchema", "true").csv("/Volumes/workspace/default/first_volume/products.csv")

####Step 2: Explore the data

In [0]:
display(customers)
display(sales)
display(products)

customers.printSchema()
sales.printSchema()
products.printSchema()

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,Raymond,Oneal,kphelps@example.org,4044671960,94488 Ronald Lights,Taylorhaven,OK,46917
2,Audrey,Vazquez,douglasrebecca@example.org,809-471-4260x4369,7505 Barbara Spurs Apt. 860,Leemouth,KY,76665
3,James,Wood,kevin10@example.org,001-340-849-7198x33540,955 Susan Court Suite 075,East Tyler,MA,12404
4,Jessica,Harris,amberbaldwin@example.org,001-975-357-4132x8668,17793 Rodriguez Junctions,Port Jason,TX,56294
5,Catherine,Benson,dianenolan@example.org,001-323-448-1037x508,4282 Jacobs Forest Suite 781,Port Thomasborough,OK,71566
6,David,Daniels,vpayne@example.net,597-841-3251,994 Alexis Walk,South Sandra,LA,26825
7,Patrick,Decker,destiny50@example.net,001-245-887-8761,3157 Joseph Ridges,Joelhaven,NE,91251
8,Jennifer,Zavala,richardwatts@example.org,(587)983-0116,46306 Michael Glens,Johnsonland,NC,73755
9,Patrick,Chavez,johnsonelizabeth@example.net,(693)982-2653x3946,3347 Vickie Summit,Anthonyville,MN,76636
10,Matthew,Phelps,sgreer@example.net,(371)904-9027x86305,7612 Christy Track,East Scottshire,FM,51870


sale_id,customer_id,product_id,sale_date,quantity,total_amount
1,69,105,2025-11-23,1,492.77
2,23,110,2025-12-24,2,780.68
3,50,109,2025-08-04,1,97.91
4,17,119,2025-10-07,1,499.87
5,15,112,2026-06-21,2,74.12
6,59,114,2026-06-30,3,795.96
7,87,118,2026-03-26,5,845.25
8,57,121,2026-01-25,1,574.38
9,43,113,2025-08-21,4,3690.76
10,48,108,2025-09-18,5,2656.5


product_id,product_name,category,price,stock_quantity
101,Laptop,Electronics,193.5,424
102,Smartphone,Electronics,164.54,335
103,Tablet,Electronics,929.09,200
104,Camera,Electronics,269.99,493
105,Printer,Electronics,492.77,493
106,Speaker,Electronics,970.2,389
107,Monitor,Electronics,669.71,118
108,Router,Electronics,531.3,295
109,Webcam,Electronics,97.91,374
110,Headphones,Accessories,390.34,76


root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: integer (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_quantity: integer (nullable = true)



#### Step 3: Clean the data

In [0]:
# Remove duplicate rows
customers = customers.dropDuplicates()
# Remove null values
customers = customers.dropna(subset=["customer_id"])
sales = sales.dropna(subset=["customer_id"])
# Convert column data types
from pyspark.sql.functions import col

sales = sales.withColumn(
    "total_amount",
    col("total_amount").cast("double")
)

#### Step 4: Transform the data

In [0]:
# Total sales per customer
from pyspark.sql.functions import sum

customer_sales = sales.groupBy("customer_id").agg(
    sum("total_amount").alias("total_spend")
)
# Join customer details
final_df = customers.join(
    customer_sales,
    "customer_id",
    "left"
)
# Fill missing values
final_df = final_df.fillna({
    "total_spend": 0
})

In [0]:
spark.sql("create catalog if not exists phase3_catalog")
spark.sql("create schema if not exists phase3_catalog.silver")

DataFrame[]

#### Step 5: Write the cleaned data

In [0]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("phase3_catalog.silver.customer_sales")

## Clean Transform using Sql

In [0]:

customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
sales.createOrReplaceTempView("sales")

#### Remove records with null customer_id

In [0]:
%sql
select *
from customers
where customer_id is not null;

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
1,Raymond,Oneal,kphelps@example.org,4044671960,94488 Ronald Lights,Taylorhaven,OK,46917
2,Audrey,Vazquez,douglasrebecca@example.org,809-471-4260x4369,7505 Barbara Spurs Apt. 860,Leemouth,KY,76665
3,James,Wood,kevin10@example.org,001-340-849-7198x33540,955 Susan Court Suite 075,East Tyler,MA,12404
4,Jessica,Harris,amberbaldwin@example.org,001-975-357-4132x8668,17793 Rodriguez Junctions,Port Jason,TX,56294
5,Catherine,Benson,dianenolan@example.org,001-323-448-1037x508,4282 Jacobs Forest Suite 781,Port Thomasborough,OK,71566
6,David,Daniels,vpayne@example.net,597-841-3251,994 Alexis Walk,South Sandra,LA,26825
7,Patrick,Decker,destiny50@example.net,001-245-887-8761,3157 Joseph Ridges,Joelhaven,NE,91251
8,Jennifer,Zavala,richardwatts@example.org,(587)983-0116,46306 Michael Glens,Johnsonland,NC,73755
9,Patrick,Chavez,johnsonelizabeth@example.net,(693)982-2653x3946,3347 Vickie Summit,Anthonyville,MN,76636
10,Matthew,Phelps,sgreer@example.net,(371)904-9027x86305,7612 Christy Track,East Scottshire,FM,51870


#### Remove duplicate customers

In [0]:
%sql
select distinct *
from customers;

customer_id,first_name,last_name,email,phone_number,address,city,state,zip_code
16,Andrea,Levy,carly91@example.com,001-399-305-2578x70504,70138 Melanie Corner,Lauraland,WI,31475
32,Scott,Kennedy,thompsonchristian@example.org,2102625069,161 Tyler Mission Suite 380,West Robertport,FM,50046
63,Madison,Mitchell,hallsherry@example.com,001-668-765-0011x141,5864 Mason Viaduct Suite 293,East Carla,AZ,92857
66,Frank,Russell,murraybrittney@example.org,911.315.4489x58533,44280 Christopher Passage Apt. 123,Andrewmouth,NY,15248
75,Angela,Lawrence,marycunningham@example.net,7498922485,539 John Course Suite 968,Hartville,KY,95292
96,Phillip,Patel,kenneth91@example.com,(218)749-9242x89262,245 Fleming Corners Apt. 569,Seanmouth,MI,39059
24,Jennifer,Oneal,avasquez@example.org,(907)635-5346,138 Douglas Isle Suite 715,Davidchester,ME,54858
29,Wesley,Rodriguez,hamiltonkeith@example.net,478.385.9399x5821,8915 Jacob Stream,West Stephanie,VT,41822
31,Olivia,Jones,hannah61@example.com,771.919.8555x503,506 Fox Corners,Leblancfort,WV,27065
51,Bruce,Mitchell,stephaniecummings@example.com,(498)792-7485x02732,3916 Kimberly Meadow,Briggsstad,DE,44289


#### Join customers, sales, and products

In [0]:
%sql
select
    c.customer_id,
    c.first_name,
    c.last_name,
    p.product_name,
    p.category,
    s.quantity,
    s.total_amount
from customers c
join sales s
on c.customer_id = s.customer_id
join products p
on s.product_id = p.product_id;

customer_id,first_name,last_name,product_name,category,quantity,total_amount
69,William,Pace,Printer,Electronics,1,492.77
23,Shawn,Harris,Headphones,Accessories,2,780.68
50,Paul,Washington,Webcam,Electronics,1,97.91
17,Collin,Neal,Vacuum Cleaner,Home,1,499.87
15,Phillip,Norton,Keyboard,Accessories,2,74.12
59,Keith,Garcia,Smartwatch,Accessories,3,795.96
87,David,Fuller,Microwave,Home,5,845.25
57,Bethany,Santiago,Trimmer,Beauty,1,574.38
43,Sarah,Graham,Charger,Accessories,4,3690.76
48,Brittany,Hill,Router,Electronics,5,2656.5


#### Total sales for each product

In [0]:
%sql
select
    product_id,
    sum(total_amount) as total_sales
from sales
group by product_id;

product_id,total_sales
105,26609.58
121,31590.899999999998
108,14876.399999999998
123,28240.489999999994
111,22473.719999999998
104,10799.6
124,14054.820000000002
109,2839.3900000000003
115,28736.120000000003
114,10612.8


#### Total quantity sold for each product

In [0]:
%sql
select
    product_id,
    sum(quantity) as total_quantity
from sales
group by product_id;

product_id,total_quantity
105,54
121,55
108,28
123,29
111,27
104,40
124,39
109,29
115,34
114,40


#### Category-wise revenue


In [0]:
%sql
select
    p.category,
    sum(s.total_amount) as revenue
from sales s
join products p
on s.product_id = p.product_id
group by p.category;

category,revenue
Home,85930.16000000002
Beauty,116334.62999999998
Electronics,171449.24000000005
Accessories,86012.96000000002


#### Top 5 selling products

In [0]:
%sql
select
    p.product_name,
    sum(s.quantity) as total_quantity
from sales s
join products p
on s.product_id = p.product_id
group by p.product_name
order by total_quantity desc
limit 5;

product_name,total_quantity
Microwave,64
Speaker,57
Trimmer,55
Printer,54
Vacuum Cleaner,52
